# Advisor Workflow — n8n Batch Runner

Sends each row of `advisor_test_case.xlsx` to the **Advisor Workflow** webhook and appends
`result_consultAcc`, `result_DebtSituation`, `result_maxPayment`, `result_maxTerm`,
`result_refPlanID`, `result_maxPaymentY2`, `result_maxPaymentY3`, and `result_reConfirmMessage` columns.
Output is saved to `advisor_result.xlsx`.

> The test-case file already stores payload fields in the **flattened** shape the webhook expects,
> so each row maps directly to the request body without further transformation.

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
N8N_BASE_URL  = "https://alphamakeathon-automation.arisetech.dev"
WEBHOOK_PATH  = "b7607735-bac3-4965-8c68-2ed0ef38aec4"
USE_TEST_URL  = False   # True → /webhook-test/…  |  False → /webhook/…

INPUT_FILE    = "advisor_test_case.xlsx"
OUTPUT_FILE   = "advisor_result.xlsx"

TIMEOUT       = 60      # seconds per request (advisor agent is slower)
RETRIES       = 2       # extra attempts on failure
DELAY_BETWEEN = 0.5     # seconds between rows

In [2]:
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

In [3]:
def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


def _str(val, default: str = "") -> str:
    return str(val) if pd.notna(val) else default


def _num(val, default=0):
    try:
        return float(val) if pd.notna(val) else default
    except (TypeError, ValueError):
        return default


def build_payload(row: dict) -> dict:
    """Map a spreadsheet row to the flat webhook payload."""
    return {
        "consultAcc":    _str(row.get("consultAcc")),
        "DebtSituation": _str(row.get("DebtSituation")),
        "maxPayment":    _num(row.get("maxPayment")),
        "maxTerm":       _num(row.get("maxTerm")),
        "refPlanID":     _str(row.get("refPlanID")),
        "maxPaymentY2":  _num(row.get("maxPaymentY2")),
        "maxPaymentY3":  _num(row.get("maxPaymentY3")),
        "narrative":     _str(row.get("narrative")),
        "accText":       _str(row.get("accText")),
        "offerText":     _str(row.get("offerText")),
        "accInfotoExtr": _str(row.get("accInfotoExtr")),
        "LastAImessage": _str(row.get("LastAImessage")),
        "userMessage":   _str(row.get("userMessage")),
    }


def call_advisor(row: dict, timeout: int = TIMEOUT, retries: int = RETRIES) -> dict:
    payload  = build_payload(row)
    url      = get_webhook_url()
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(url, json=payload, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last_exc


print("Webhook URL:", get_webhook_url())

Webhook URL: https://alphamakeathon-automation.arisetech.dev/webhook/b7607735-bac3-4965-8c68-2ed0ef38aec4


In [4]:
df = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df)} rows  |  columns: {list(df.columns)}")
df.head(3)

Loaded 250 rows  |  columns: ['testId', 'userMessage', 'LastAImessage', 'accText', 'offerText', 'consultAcc', 'DebtSituation', 'maxPayment', 'maxTerm', 'refPlanID', 'maxPaymentY2', 'maxPaymentY3', 'narrative', 'expected_consultAcc', 'expected_DebtSituation', 'expected_maxPayment', 'expected_maxTerm', 'expected_refPlanID', 'expected_maxPaymentY2', 'expected_maxPaymentY3', 'expected_reConfirmMessage']


,testId,userMessage,LastAImessage,accText,offerText,consultAcc,DebtSituation,maxPayment,maxTerm,refPlanID,...,maxPaymentY3,narrative,expected_consultAcc,expected_DebtSituation,expected_maxPayment,expected_maxTerm,expected_refPlanID,expected_maxPaymentY2,expected_maxPaymentY3,expected_reConfirmMessage
0,TC-0001,รายได้ต่อเดือนยังเท่าเดิมแต่ภาระหนี้รวมเยอะมาก...,น้องฟินขอทราบบัญชีที่ต้องการปรับโครงสร้างหนี้ค...,"[{""account_number"": ""xxxxxx500007"", ""port"": ""ส...",NaN,NaN,NaN,0,360,NaN,...,NaN,New customer request; no previous plan has bee...,"xxxxxx500007,xxxxxx500008,xxxxxx500009",DebtBurden,5000,36,NaN,NaN,NaN,NaN
1,TC-0002,ตอนนี้หนี้หลายทางเกินรายรับ อยากลดค่างวดลงให้พ...,สวัสดีค่ะ กรุณาแจ้งข้อมูลหนี้และกำลังผ่อนที่ต้...,"[{""account_number"": ""xxxxxx500014"", ""port"": ""ส...",NaN,NaN,NaN,0,360,NaN,...,NaN,New customer request; no previous plan has bee...,xxxxxx500015,DebtBurden,15000,120,NaN,NaN,NaN,NaN
2,TC-0003,ค่าใช้จ่ายประจำสูงจนเงินเดือนเหลือไม่พอผ่อนเต็...,สวัสดีค่ะ กรุณาแจ้งข้อมูลหนี้และกำลังผ่อนที่ต้...,"[{""account_number"": ""xxxxxx500021"", ""port"": ""ส...",NaN,NaN,NaN,0,360,NaN,...,NaN,New customer request; no previous plan has bee...,xxxxxx500023,DebtBurden,20000,84,NaN,NaN,NaN,NaN


In [5]:
RESULT_FIELDS = [
    "consultAcc", "DebtSituation", "maxPayment", "maxTerm",
    "refPlanID", "maxPaymentY2", "maxPaymentY3", "reConfirmMessage",
]

result_buckets = {f: [] for f in RESULT_FIELDS}
errors = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Calling n8n"):
    try:
        resp = call_advisor(row.to_dict())
        for field in RESULT_FIELDS:
            result_buckets[field].append(resp.get(field, ""))
        errors.append(None)
    except Exception as exc:
        for field in RESULT_FIELDS:
            result_buckets[field].append(None)
        errors.append(str(exc))
    time.sleep(DELAY_BETWEEN)

for field in RESULT_FIELDS:
    df[f"result_{field}"] = result_buckets[field]
df["error"] = errors

failed = df["error"].notna().sum()
print(f"Done. {len(df) - failed}/{len(df)} succeeded, {failed} failed.")

Calling n8n:   0%|          | 0/250 [00:00<?, ?it/s]

Done. 250/250 succeeded, 0 failed.


In [6]:
# Preview failed rows (if any)
if failed:
    display(df[df["error"].notna()][["testId", "userMessage", "error"]])

In [7]:
df.to_excel(OUTPUT_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")
result_cols = [f"result_{f}" for f in RESULT_FIELDS]
df[["testId"] + result_cols].head(5)

Saved → advisor_result.xlsx


,testId,result_consultAcc,result_DebtSituation,result_maxPayment,result_maxTerm,result_refPlanID,result_maxPaymentY2,result_maxPaymentY3,result_reConfirmMessage
0,TC-0001,"xxxxxx500007,xxxxxx500008,xxxxxx500009",DebtBurden,5000,36,None,None,None,
1,TC-0002,xxxxxx500015,DebtBurden,15000,120,None,None,None,
2,TC-0003,xxxxxx500023,DebtBurden,20000,84,None,None,None,
3,TC-0004,"xxxxxx500028,xxxxxx500029,xxxxxx500030",DebtBurden,30000,180,None,None,None,
4,TC-0005,xxxxxx500035,DebtBurden,4500,36,None,None,None,
